# X-Ray Blue — defend the active Red round

The launcher placed this round's successful Red examples in your personal `xray_blue` folder. These examples are for inspection and local experiments. The protected evaluator scores your defense on the **full frozen batch**, including images the Red attack did not flip. Try the supplied standard and robust checkpoints, preprocessing, and detection options. Submit a defense configuration when ready.


In [ ]:
import json
import os
import time
from hashlib import sha256
from pathlib import Path

import numpy as np
import requests
from IPython.display import display
from PIL import Image, ImageDraw

BLUE_DIR = Path('/workspace/xray_blue')
manifest = json.loads((BLUE_DIR / 'active_round.json').read_text(encoding='utf-8'))
artifact = BLUE_DIR / manifest['artifact']
if sha256(artifact.read_bytes()).hexdigest() != manifest['sha256']:
    raise RuntimeError('The Red example package is incomplete. Reopen X-Ray Blue from CTFd.')
with np.load(artifact, allow_pickle=False) as red_data:
    clean_images = red_data['x_clean'].copy()
    red_images = red_data['x_adv'].copy()
    labels = red_data['labels'].copy()
    clean_predictions = red_data['clean_predictions'].copy()
    adv_predictions = red_data['adv_predictions'].copy()
if not len(labels):
    raise RuntimeError('No successful Red examples were published for this round.')
print('Active round:', manifest['round_id'])
print('Successful Red examples:', len(labels))


In [ ]:
def show_red_example(index=0):
    if not 0 <= index < len(labels):
        raise IndexError('Choose an index among the successful Red examples.')
    left = np.rint(np.clip(clean_images[index].transpose(1, 2, 0), 0, 1) * 255).astype('uint8')
    right = np.rint(np.clip(red_images[index].transpose(1, 2, 0), 0, 1) * 255).astype('uint8')
    canvas = Image.new('RGB', (left.shape[1] * 2, left.shape[0] + 28), 'white')
    canvas.paste(Image.fromarray(left), (0, 28))
    canvas.paste(Image.fromarray(right), (left.shape[1], 28))
    draw = ImageDraw.Draw(canvas)
    draw.text((8, 8), 'Clean', fill='black')
    draw.text((left.shape[1] + 8, 8), 'Red adversarial', fill='black')
    display(canvas)
    print('True label:', labels[index], 'clean prediction:', clean_predictions[index], 'Red prediction:', adv_predictions[index])

show_red_example(0)


## Evaluate a defense

Select a model (`standard` or `robust`), a preprocessor (`none`, `median3`, `gaussian`, or `quantize`), and a detector (`none`, `consistency`, or `disagreement`). The evaluator reports clean utility, adversarial resolution, detection, and false positives. A passing result includes the Blue flag for CTFd. The shared round accepts one passing Blue defense.


In [ ]:
XRAY_URL = os.getenv('XRAY_REDBLUE_URL', 'http://xray-redblue:5000').rstrip('/')
def xray_headers():
    token = os.getenv('ARENA_TOKEN', '')
    return {'Authorization': f'Bearer {token}'} if token else {}

def evaluate_blue(*, model='robust', preprocess='none', bits=5, detector='none', samples=4, sigma=0.03, threshold=0.75, confidence=0.6):
    current = requests.get(f'{XRAY_URL}/api/round', headers=xray_headers(), timeout=20)
    current.raise_for_status()
    if current.json()['round_id'] != manifest['round_id']:
        raise RuntimeError('Your saved Red examples are from an earlier round. Reopen X-Ray Blue from CTFd.')
    params = {'model': model, 'preprocess': preprocess, 'bits': bits, 'detector': detector, 'samples': samples, 'sigma': sigma, 'threshold': threshold, 'confidence': confidence}
    response = requests.post(f'{XRAY_URL}/api/blue/defend', json=params, headers=xray_headers(), timeout=20)
    response.raise_for_status()
    job_url = XRAY_URL + response.json()['poll_url']
    deadline = time.monotonic() + 300
    while time.monotonic() < deadline:
        job = requests.get(job_url, headers=xray_headers(), timeout=20)
        job.raise_for_status()
        status = job.json()
        if status['status'] == 'failed':
            raise RuntimeError(status.get('error', 'Blue evaluation failed'))
        if status['status'] == 'completed':
            result = status['result']
            print('Success:', result['success'])
            for key in ('clean_usable_accuracy', 'adv_resolved_accuracy', 'detection_rate', 'false_positive_rate'):
                print(f'{key}: {result[key]:.4f}')
            if result.get('flag'):
                print('Submit this flag in CTFd:', result['flag'])
            return result
        time.sleep(1)
    raise TimeoutError('Blue evaluation did not finish within five minutes. Check the job URL before retrying.')


In [ ]:
# TODO: choose and explain your own defense configuration.
# Example call: blue_result = evaluate_blue(model='robust', detector='consistency', samples=8, sigma=0.01, threshold=1.0)
